In [ ]:
## Load Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import geopandas as gpd
import seaborn as sns
from libpysal.weights import DistanceBand, KNN
from esda.moran import Moran
from spreg import OLS, ML_Error, ML_Lag, GM_Error, GM_Lag
from scipy import stats
from shapely.geometry import Point
from splot.esda import moran_scatterplot, plot_moran
import contextily as ctx
from matplotlib.ticker import FuncFormatter
import pyproj
from libpysal.weights import lag_spatial
from scipy.stats import norm

In [ ]:
# Load data
ed_listings = pd.read_csv('data/listings.csv')

#### To select the variables I will use a rule-based approach:
1. In my view, is this variable relevant to listing price?
2. Is it missing too much data for it to be useful?
3. Is it redundant or share strong similarities to other variables?
4. Is it a leakage variable? A variable that is only known once price has been defined.
5. Is it a URL or any type of data that is not numerical or categorical that I can use without the need to transform it?

In [ ]:
ed_listings.columns

In [ ]:
all_amenities = ed_listings['amenities'].str.strip('[]').str.replace('"', '').str.split(',').explode().str.strip()

amenities_df = all_amenities.value_counts().reset_index()
amenities_df.columns = ['amenity', 'count']

#pd.set_option('display.max_rows', None)
amenities_df[amenities_df['count'] >= 100]
#pd.reset_option('display.max_rows')

In [ ]:
# Dataframe containing all variables and data type
df_description = pd.DataFrame({'variable_names': ed_listings.columns})
df_description['data_type'] = ed_listings.dtypes.values
print(df_description)


In [ ]:
print(ed_listings['property_type'].unique())

#### Creating binary/dummy variables from 'ammenities' using my criteria and then applying the following rule of thumb:
- If its too common (>70%) or too rare (<10%) I will drop it.
<br><br>
This leaves me with the following ammenities which I will use as 'dummies':
- dedicated workspace
- indoor fireplace
- patio or balcony


In [ ]:
# Variables of interest based on applied criteria


In [ ]:
# ---- One-hot encoding for neighborhood variable

ed_listings = pd.get_dummies(ed_listings, columns=['neighbourhood_cleansed'], drop_first=True)

# drop_first=True automatically drops one neighbourhood as the reference category that all others are compared against. 
# The coefficients then tell you how much more or less expensive each neighbourhood is relative to that baseline.

| Variable                              | Type              | Rationale                                                                 |
|---------------------------------------|-------------------|---------------------------------------------------------------------------|
| price                                 | Dependent (Y)     | Daily listing price in local currency (log-transform recommended)         |
| host_is_superhost                     | Host              | Quality signal — superhost status may command a price premium             |
| calculated_host_listings_count        | Host              | Professional vs individual host — multi-property operators price differently |
| room_type                             | Property          | Entire place vs private vs shared room — strongest price-driving distinction (dummies) |
| accommodates                          | Property          | Max guest capacity — directly linked to price                             |
| bathrooms                             | Property          | Number of bathrooms — premium for more bathrooms                          |
| bedrooms                              | Property          | Number of bedrooms — key size indicator                                   |
| has_dedicated_workspace               | Property          | Dummy from amenities — appeals to business/longer-stay guests             |
| has_indoor_fireplace                  | Property          | Dummy from amenities — luxury/character signal in Edinburgh's historic properties |
| has_patio_or_balcony                  | Property          | Dummy from amenities — outdoor space premium                              |
| minimum_nights                        | Policy            | Minimum stay requirement — may reflect listing strategy                   |
| maximum_nights                        | Policy            | Maximum stay cap — keep for now, may be insignificant                     |
| number_of_reviews                     | Demand            | Proxy for popularity and demand                                           |
| availability_365                      | Demand            | Annual availability — reflects how competitive/desirable the listing is   |
| review_scores_rating                  | Demand            | Overall guest rating as a quality indicator                               |
| reviews_per_month                     | Demand            | Booking activity rate — busier listings may price differently             |
| instant_bookable                      | Policy            | Boolean — convenience feature that may influence pricing strategy         |
| neighbourhood_cleansed_*              | Location          | One-hot encoded dummies for each neighbourhood (already created)          |
| latitude                              | Spatial           | Required for spatial models and GWR — not a regression predictor          |
| longitude                             | Spatial           | Required for spatial models and GWR — not a regression predictor          |

| Variable                | Type             | Rationale |
| ----------              | -------          |-----------
| price                   | Dependent (Y)    | Daily price in local currency
| host_is_superhost       | Host             | >= 4.8 rating, >= 100 nights annually, <1% cancellations
| neighbourhood           | Property         | Neighbourhood of the accommodation
| room_type               | Property         | Entire place, private room, or shared room
| accommodates            | Property         | Max capacity of the listing
| bathrooms               | Property         | Number of bathrooms
| bedrooms                | Property         | Number of bedrooms
| has_dedicated_workplace | Property         | Dummy variable taken from 'amenities'
| has_indoor_fireplace    | Property         | Dummy variable taken from 'amenities'
| has_patio_or_balcony    | Property         | Dummy variable taken from 'amenities'
| minimum_nights          | Policy           | Minimum number of night stay for the listing
| maximum_nights          | Policy           | Maximum number of night stay for the listing
| review_scores_rating    | Demand           | Overall guest rating as a quality indicator
| instant_bookable        | Policy           | Whether the guest can automatically book the listing without the host having to accept
| number_of_reviews       | Property         | Proxy for demand/popularity
| reviews_per_month       | Property         | Captures how actively booked a listing is